# CultureConnect - Data Analysis

This notebook loads the CultureConnect database and runs SQL queries to explore the data.

The database covers users, areas, categories, listings, bookings, orders and reviews for a community culture platform in Hertfordshire.

In [ ]:
!pip install -q pandas matplotlib seaborn

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded')

## Step 1 - Set Up the Database

We create an in-memory SQLite database and load in all the tables and data from the CultureConnect schema.

In [ ]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create tables
cursor.executescript('''
CREATE TABLE areas (
    area_id INTEGER PRIMARY KEY,
    area_name TEXT,
    description TEXT,
    postcode TEXT,
    latitude REAL,
    longitude REAL
);

CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT,
    description TEXT
);

CREATE TABLE roles (
    role_id INTEGER PRIMARY KEY,
    role_name TEXT
);

CREATE TABLE users (
    user_ref_no INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT,
    role_id INTEGER,
    area_id INTEGER,
    status TEXT,
    created_at TEXT,
    user_code TEXT
);

CREATE TABLE sme_businesses (
    sme_id INTEGER PRIMARY KEY,
    user_ref_no INTEGER,
    business_name TEXT,
    category TEXT,
    location TEXT,
    approval_status TEXT,
    created_at TEXT
);

CREATE TABLE listings (
    listing_id INTEGER PRIMARY KEY,
    sme_id INTEGER,
    title TEXT,
    category_id INTEGER,
    price REAL,
    type TEXT,
    status TEXT,
    created_at TEXT
);

CREATE TABLE service_bookings (
    booking_id INTEGER PRIMARY KEY,
    user_ref_no INTEGER,
    listing_id INTEGER,
    booking_date TEXT,
    status TEXT
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    user_ref_no INTEGER,
    order_date TEXT,
    total_amount REAL,
    status TEXT
);

CREATE TABLE reviews (
    review_id INTEGER PRIMARY KEY,
    user_ref_no INTEGER,
    listing_id INTEGER,
    rating INTEGER,
    comment TEXT,
    created_at TEXT
);
''')

print('Tables created')

In [ ]:
# Load the real data from the schema

cursor.executescript('''
-- Areas
INSERT INTO areas VALUES (1, 'Hertfordshire North', 'Famous for visual arts', 'AL10 9NA', 51.7632, -0.2381);
INSERT INTO areas VALUES (2, 'Hertfordshire South', 'Famous for music and performing arts', 'AL10 9SB', 51.7487, -0.2403);
INSERT INTO areas VALUES (3, 'Hertfordshire East', 'Famous for creative media', 'AL10 0ED', 51.7599, -0.2256);
INSERT INTO areas VALUES (4, 'Hertfordshire West', 'Famous for literature and publishing', 'AL10 0WB', 51.7615, -0.2574);
INSERT INTO areas VALUES (5, 'Hertfordshire Town Centre', 'Known for cultural markets', 'AL10 0JT', 51.7606, -0.2423);
INSERT INTO areas VALUES (6, 'Hertfordshire Central', 'Famous for sports and recreation', 'AL10 8HG', 51.7654, -0.2207);

-- Categories
INSERT INTO categories VALUES (1, 'Visual Arts', 'Painting, sculpture, ceramics and art workshops');
INSERT INTO categories VALUES (2, 'Music', 'Music lessons, concerts, and musical training');
INSERT INTO categories VALUES (3, 'Performing Arts', 'Theatre productions, spoken word, and stage performances');
INSERT INTO categories VALUES (4, 'Creative Media', 'Photography, videography, and digital design');
INSERT INTO categories VALUES (5, 'Literature', 'Books, poetry, and creative writing');
INSERT INTO categories VALUES (6, 'Cultural Merchandise', 'Posters, handmade crafts, and artisan stationery');
INSERT INTO categories VALUES (7, 'Sports & Recreation', 'Community sports and cultural recreation');
INSERT INTO categories VALUES (8, 'Wellbeing & Community', 'Art therapy, music therapy, and wellbeing workshops');

-- Roles
INSERT INTO roles VALUES (1, 'Resident');
INSERT INTO roles VALUES (2, 'SME');
INSERT INTO roles VALUES (3, 'Council Member');
INSERT INTO roles VALUES (4, 'Council Administrator');

-- Users (from schema)
INSERT INTO users VALUES (1, 'Oyin Alade', 'resident1@example.com', 1, 1, 'active', '2026-03-15 18:23:03', 'RES-0001');
INSERT INTO users VALUES (2, 'Josephine Abioye', 'sme1@example.com', 2, 1, 'approved', '2026-03-15 18:23:03', 'SME-0001');
INSERT INTO users VALUES (3, 'Victor Ehizefua', 'council1@example.com', 3, 5, 'approved', '2026-03-15 18:23:03', 'CNS-0001');
INSERT INTO users VALUES (4, 'Council Admin', 'admin@example.com', 4, 3, 'approved', '2026-03-15 18:23:03', 'ADM-0001');
INSERT INTO users VALUES (5, 'Tunde Balogun', 'resident2@example.com', 1, 1, 'active', '2026-03-15 19:01:42', 'RES-0002');
INSERT INTO users VALUES (6, 'Amara Osei', 'resident3@example.com', 1, 2, 'active', '2026-03-20 10:15:00', 'RES-0003');
INSERT INTO users VALUES (7, 'Funmi Adeyemi', 'sme2@example.com', 2, 3, 'approved', '2026-03-22 09:00:00', 'SME-0002');
INSERT INTO users VALUES (8, 'Chidi Okafor', 'resident4@example.com', 1, 4, 'active', '2026-03-25 14:30:00', 'RES-0004');
INSERT INTO users VALUES (9, 'Ngozi Eze', 'sme3@example.com', 2, 2, 'pending', '2026-03-28 11:00:00', 'SME-0003');
INSERT INTO users VALUES (10, 'Emeka Nwosu', 'resident5@example.com', 1, 5, 'active', '2026-04-01 08:45:00', 'RES-0005');
INSERT INTO users VALUES (11, 'Bisi Lawson', 'resident6@example.com', 1, 6, 'active', '2026-04-03 13:20:00', 'RES-0006');
INSERT INTO users VALUES (12, 'Kwame Mensah', 'sme4@example.com', 2, 4, 'approved', '2026-04-05 10:00:00', 'SME-0004');
INSERT INTO users VALUES (13, 'Sade Okonkwo', 'resident7@example.com', 1, 3, 'active', '2026-04-08 15:00:00', 'RES-0007');
INSERT INTO users VALUES (14, 'Tobi Adesanya', 'resident8@example.com', 1, 1, 'inactive', '2026-04-10 09:30:00', 'RES-0008');
INSERT INTO users VALUES (15, 'Ife Bello', 'sme5@example.com', 2, 6, 'approved', '2026-04-12 11:45:00', 'SME-0005');

-- SME Businesses
INSERT INTO sme_businesses VALUES (1, 2, 'Josephine Art Studio', 'Visual Arts', 'Hatfield', 'approved', '2026-03-15');
INSERT INTO sme_businesses VALUES (2, 7, 'Funmi Music School', 'Music', 'Hatfield', 'approved', '2026-03-22');
INSERT INTO sme_businesses VALUES (3, 9, 'Ngozi Photography', 'Creative Media', 'Hatfield', 'pending', '2026-03-28');
INSERT INTO sme_businesses VALUES (4, 12, 'Kwame Craft Works', 'Cultural Merchandise', 'Hatfield', 'approved', '2026-04-05');
INSERT INTO sme_businesses VALUES (5, 15, 'Ife Wellness Centre', 'Wellbeing & Community', 'Hatfield', 'approved', '2026-04-12');

-- Listings
INSERT INTO listings VALUES (1, 1, 'Watercolour Painting Workshop', 1, 45.00, 'service', 'active', '2026-03-16');
INSERT INTO listings VALUES (2, 1, 'Pottery Class for Beginners', 1, 35.00, 'service', 'active', '2026-03-17');
INSERT INTO listings VALUES (3, 1, 'Portrait Drawing Session', 1, 50.00, 'service', 'active', '2026-03-18');
INSERT INTO listings VALUES (4, 2, 'Guitar Lessons for Beginners', 2, 30.00, 'service', 'active', '2026-03-23');
INSERT INTO listings VALUES (5, 2, 'Drum and Percussion Class', 2, 40.00, 'service', 'active', '2026-03-24');
INSERT INTO listings VALUES (6, 2, 'Music Theory Workshop', 2, 25.00, 'service', 'active', '2026-03-25');
INSERT INTO listings VALUES (7, 4, 'Handmade Ceramic Mugs', 6, 18.00, 'product', 'active', '2026-04-06');
INSERT INTO listings VALUES (8, 4, 'Artisan Greeting Cards', 6, 5.00, 'product', 'active', '2026-04-07');
INSERT INTO listings VALUES (9, 5, 'Mindfulness and Art Session', 8, 20.00, 'service', 'active', '2026-04-13');
INSERT INTO listings VALUES (10, 5, 'Community Wellbeing Workshop', 8, 15.00, 'service', 'active', '2026-04-14');
INSERT INTO listings VALUES (11, 1, 'Abstract Art Masterclass', 1, 60.00, 'service', 'inactive', '2026-04-01');
INSERT INTO listings VALUES (12, 2, 'Choir and Vocal Training', 2, 35.00, 'service', 'active', '2026-04-15');

-- Service Bookings
INSERT INTO service_bookings VALUES (1, 1, 1, '2026-04-01', 'completed');
INSERT INTO service_bookings VALUES (2, 5, 2, '2026-04-02', 'completed');
INSERT INTO service_bookings VALUES (3, 6, 4, '2026-04-03', 'completed');
INSERT INTO service_bookings VALUES (4, 8, 5, '2026-04-05', 'completed');
INSERT INTO service_bookings VALUES (5, 10, 6, '2026-04-06', 'confirmed');
INSERT INTO service_bookings VALUES (6, 11, 9, '2026-04-08', 'completed');
INSERT INTO service_bookings VALUES (7, 13, 1, '2026-04-10', 'completed');
INSERT INTO service_bookings VALUES (8, 1, 4, '2026-04-12', 'cancelled');
INSERT INTO service_bookings VALUES (9, 5, 10, '2026-04-14', 'confirmed');
INSERT INTO service_bookings VALUES (10, 6, 3, '2026-04-15', 'completed');
INSERT INTO service_bookings VALUES (11, 8, 12, '2026-04-16', 'completed');
INSERT INTO service_bookings VALUES (12, 10, 2, '2026-04-18', 'completed');

-- Orders
INSERT INTO orders VALUES (1, 1, '2026-04-05', 45.00, 'completed');
INSERT INTO orders VALUES (2, 5, '2026-04-06', 18.00, 'completed');
INSERT INTO orders VALUES (3, 6, '2026-04-07', 35.00, 'completed');
INSERT INTO orders VALUES (4, 8, '2026-04-08', 10.00, 'completed');
INSERT INTO orders VALUES (5, 10, '2026-04-10', 60.00, 'completed');
INSERT INTO orders VALUES (6, 11, '2026-04-12', 20.00, 'pending');
INSERT INTO orders VALUES (7, 13, '2026-04-13', 40.00, 'completed');
INSERT INTO orders VALUES (8, 1, '2026-04-15', 15.00, 'completed');
INSERT INTO orders VALUES (9, 5, '2026-04-17', 50.00, 'cancelled');
INSERT INTO orders VALUES (10, 6, '2026-04-18', 25.00, 'completed');

-- Reviews
INSERT INTO reviews VALUES (1, 1, 1, 5, 'Amazing workshop, very welcoming teacher', '2026-04-03');
INSERT INTO reviews VALUES (2, 5, 2, 4, 'Really enjoyed the pottery class', '2026-04-04');
INSERT INTO reviews VALUES (3, 6, 4, 5, 'Great guitar lessons, very patient instructor', '2026-04-05');
INSERT INTO reviews VALUES (4, 8, 5, 4, 'Fun drum class, will book again', '2026-04-07');
INSERT INTO reviews VALUES (5, 11, 9, 5, 'The mindfulness session was really calming', '2026-04-09');
INSERT INTO reviews VALUES (6, 13, 1, 3, 'Good but could be longer', '2026-04-11');
INSERT INTO reviews VALUES (7, 10, 6, 4, 'Learned a lot about music theory', '2026-04-13');
INSERT INTO reviews VALUES (8, 6, 3, 5, 'Brilliant portrait session', '2026-04-16');
''')

print('Data loaded successfully')

## Step 2 - Explore the Data

We use pandas to run SQL queries and look at the data in each table.

In [ ]:
# Check what we have in each table
tables = ['areas', 'categories', 'roles', 'users', 'sme_businesses', 'listings', 'service_bookings', 'orders', 'reviews']

for table in tables:
    count = pd.read_sql(f'SELECT COUNT(*) AS total FROM {table}', conn).iloc[0, 0]
    print(f'{table}: {count} rows')

In [ ]:
# Preview the users table joined with roles and areas
query = '''
SELECT u.name, r.role_name, a.area_name, u.status
FROM users u
JOIN roles r ON u.role_id = r.role_id
JOIN areas a ON u.area_id = a.area_id
'''
pd.read_sql(query, conn)

## Step 3 - Users by Role and Area

In [ ]:
query = '''
SELECT r.role_name, COUNT(u.user_ref_no) AS total_users
FROM roles r
LEFT JOIN users u ON r.role_id = u.role_id
GROUP BY r.role_name
ORDER BY total_users DESC
'''
df = pd.read_sql(query, conn)

ax = df.plot(kind='bar', x='role_name', y='total_users', legend=False,
             color=sns.color_palette('muted'), edgecolor='white')
ax.set_title('Number of Users by Role')
ax.set_xlabel('')
ax.set_ylabel('Users')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()
print(df.to_string(index=False))

In [ ]:
query = '''
SELECT a.area_name, COUNT(u.user_ref_no) AS total_users
FROM areas a
LEFT JOIN users u ON a.area_id = u.area_id
GROUP BY a.area_name
ORDER BY total_users DESC
'''
df = pd.read_sql(query, conn)

ax = df.plot(kind='barh', x='area_name', y='total_users', legend=False,
             color=sns.color_palette('pastel'), edgecolor='white')
ax.set_title('Users per Area')
ax.set_xlabel('Users')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Step 4 - Listings by Category

In [ ]:
query = '''
SELECT c.category_name, COUNT(l.listing_id) AS total_listings, ROUND(AVG(l.price), 2) AS avg_price
FROM categories c
LEFT JOIN listings l ON c.category_id = l.category_id
GROUP BY c.category_name
ORDER BY total_listings DESC
'''
df = pd.read_sql(query, conn)

ax = df.plot(kind='bar', x='category_name', y='total_listings', legend=False,
             color=sns.color_palette('Set2'), edgecolor='white')
ax.set_title('Listings per Category')
ax.set_xlabel('')
ax.set_ylabel('Listings')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()
print(df.to_string(index=False))

## Step 5 - Bookings

In [ ]:
query = '''
SELECT c.category_name, COUNT(sb.booking_id) AS total_bookings
FROM service_bookings sb
JOIN listings l ON sb.listing_id = l.listing_id
JOIN categories c ON l.category_id = c.category_id
GROUP BY c.category_name
ORDER BY total_bookings DESC
'''
df = pd.read_sql(query, conn)

ax = df.plot(kind='bar', x='category_name', y='total_bookings', legend=False,
             color=sns.color_palette('muted'), edgecolor='white')
ax.set_title('Total Bookings per Category')
ax.set_xlabel('')
ax.set_ylabel('Bookings')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
query = '''
SELECT status, COUNT(*) AS total
FROM service_bookings
GROUP BY status
'''
df = pd.read_sql(query, conn)

colors = ['#2ecc71', '#f39c12', '#e74c3c']
plt.pie(df['total'], labels=df['status'], autopct='%1.1f%%', colors=colors, startangle=140)
plt.title('Booking Status Breakdown')
plt.tight_layout()
plt.show()

## Step 6 - Reviews and Ratings

In [ ]:
query = '''
SELECT rating, COUNT(*) AS total
FROM reviews
GROUP BY rating
ORDER BY rating
'''
df = pd.read_sql(query, conn)

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
ax = df.plot(kind='bar', x='rating', y='total', legend=False, color=colors, edgecolor='white')
ax.set_title('Review Rating Distribution')
ax.set_xlabel('Rating (1 = Poor, 5 = Excellent)')
ax.set_ylabel('Number of Reviews')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

avg = pd.read_sql('SELECT ROUND(AVG(rating), 2) AS avg_rating FROM reviews', conn)
print(f'Average rating: {avg.iloc[0, 0]} out of 5')

In [ ]:
query = '''
SELECT c.category_name, ROUND(AVG(r.rating), 2) AS avg_rating
FROM reviews r
JOIN listings l ON r.listing_id = l.listing_id
JOIN categories c ON l.category_id = c.category_id
GROUP BY c.category_name
ORDER BY avg_rating DESC
'''
df = pd.read_sql(query, conn)

ax = df.plot(kind='barh', x='category_name', y='avg_rating', legend=False,
             color=sns.color_palette('RdYlGn', len(df)), edgecolor='white')
ax.set_title('Average Rating by Category')
ax.set_xlabel('Average Rating')
ax.set_ylabel('')
ax.set_xlim(0, 5)
plt.tight_layout()
plt.show()

## Step 7 - Summary

In [ ]:
total_users = pd.read_sql('SELECT COUNT(*) AS n FROM users', conn).iloc[0, 0]
total_listings = pd.read_sql('SELECT COUNT(*) AS n FROM listings WHERE status = "active"', conn).iloc[0, 0]
total_bookings = pd.read_sql('SELECT COUNT(*) AS n FROM service_bookings', conn).iloc[0, 0]
total_revenue = pd.read_sql('SELECT SUM(total_amount) AS n FROM orders WHERE status = "completed"', conn).iloc[0, 0]
avg_rating = pd.read_sql('SELECT ROUND(AVG(rating), 2) AS n FROM reviews', conn).iloc[0, 0]

print('===== CULTURECONNECT SUMMARY =====')
print(f'Total users:      {total_users}')
print(f'Active listings:  {total_listings}')
print(f'Total bookings:   {total_bookings}')
print(f'Total revenue:    GBP {total_revenue:.2f}')
print(f'Average rating:   {avg_rating} / 5')